In [19]:
import openai
from qdrant_client import QdrantClient

In [20]:
def get_collection_name():
    return "Hackathon-Attestations-Collection-00"

In [21]:
def get_example_question():
    return "Do you have examples of projects that use attestations?"

### Embedding function

In [22]:
def get_embedding(text,model="text-embedding-3-small"):
    response=openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

### Retrieval function

In [23]:
qdrant_client=QdrantClient(url="http://localhost:6333")

def retrieve_data(query,qdrant_client,k=5):
    query_embedding=get_embedding(query)
    results=qdrant_client.query_points(
        collection_name=get_collection_name(),
        query=query_embedding,
        limit=k
    )
    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    for result in results.points:
        retrieved_context_ids.append(result.payload["id"])
        retrieved_context.append(result.payload["vectorization_data"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids":retrieved_context_ids,
        "retrieved_context":retrieved_context,
        "similarity_scores":similarity_scores
    }

retrieved_context=retrieve_data("Do you have examples of projects that use attestations?",qdrant_client,k=10)

### Format retrieved context

In [24]:
def process_context(context):
    formatted_context=""
    for id,chunk in zip(context["retrieved_context_ids"],context["retrieved_context"]):
        formatted_context+=f"- {id}: {chunk}\n"
    
    return formatted_context

print(process_context(retrieved_context))
preprocessed_context=process_context(retrieved_context)

- 0x97e56d1a1638462cedcacfc48d51dedee93d5865a5bfa9d1255f9e7df4aa4043: {project_attributes: [{trait_type: hackathon_name, value: ETHDenver 2024}, {trait_type: nft_type, value: BUILDER}, {trait_type: team_name, value: Gli Scoppiati}, {trait_type: project_name, value: Prometheus}, {display_type: date, trait_type: project_submission_date, value: 1709386196113}], project_data: {url: https://devfolio.co/projects/prometheus-ef2d, title: Prometheus, subtitle: Create an attestation of your projects and inspire others to build on them., technologies: [React, TypeScript, Vite, Wagmi, MetaMask SDK, Web3Onboard, Verax SDK], matching_amount_usd: null, votes: 419, quadratic_votes: 51.181, built_at: ETHDenver 2024 Best Attestation App with Verax, created_on: 2nd March 2024, last_edited: 2nd March 2024, problem_statement: A lot of hackathon projects are often born and die very quickly, using Prometheus you can fast create an attestation of your project and you can inspire others to build on top of them

### Create prompt function

In [25]:
def build_prompt(preprocessed_context,question):

    prompt= f"""
You are a shopping assistant that can answer questions about hte product in stock.
You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt

In [26]:
prompt=build_prompt(preprocessed_context,"What kind of earphones can I get?")
print(prompt)


You are a shopping assistant that can answer questions about hte product in stock.
You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- 0x97e56d1a1638462cedcacfc48d51dedee93d5865a5bfa9d1255f9e7df4aa4043: {project_attributes: [{trait_type: hackathon_name, value: ETHDenver 2024}, {trait_type: nft_type, value: BUILDER}, {trait_type: team_name, value: Gli Scoppiati}, {trait_type: project_name, value: Prometheus}, {display_type: date, trait_type: project_submission_date, value: 1709386196113}], project_data: {url: https://devfolio.co/projects/prometheus-ef2d, title: Prometheus, subtitle: Create an attestation of your projects and inspire others to build on them., technologies: [React, TypeScript, Vite, Wagmi, MetaMask SDK, Web3Onboard, Verax SDK], matching_amount_usd: null, votes: 419, quadratic_votes: 51.181, built_at: ETHDenve

### Generate Answer function

In [27]:
def generate_answer(prompt):

    response=openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":prompt}]
    )
    return response.choices[0].message.content

print(generate_answer(prompt))

The available products do not include any earphones. If you have any other questions or need information on different products, feel free to ask!


### Combined RAG Pipeline

In [28]:
def rag_pipeline(question,top_k=5):
    qdrant_client=QdrantClient(url="http://localhost:6333")

    retrieved_context=retrieve_data(question,qdrant_client,top_k)
    preprocessed_context=process_context(retrieved_context)
    prompt=build_prompt(preprocessed_context,question)
    answer=generate_answer(prompt)
    return answer

In [29]:
print(rag_pipeline(get_example_question(),top_k=10))

Yes, here are examples of projects that use attestations:

1. **Prometheus**
   - **Team Name:** Gli Scoppiati
   - **Description:** Create an attestation of your projects and inspire others to build on them.
   - **Technologies:** React, TypeScript, Vite, Wagmi, MetaMask SDK, Web3Onboard, Verax SDK
   - [GitHub Link](https://github.com/mmatteo23/ethdenver-prometeus)
   - [Project URL](https://devfolio.co/projects/prometheus-ef2d)

2. **EBF Network of Trust**
   - **Team Name:** ADBUTH
   - **Description:** A discoverable network of trust for registering impact projects and verifying their ecological benefits, built using verax attestations.
   - **Technologies:** IPFS, nextjs, Hardhat, ERC1155, NFT.Storage, Linea, verax, Phosphor, ScaffoldETH2, Gitcoin Passport
   - [GitHub Link](https://github.com/bhargavkakadiya/eth-denver-ebf)
   - [Project URL](https://devfolio.co/projects/trust-impact-4bf0)

These projects utilize attestations to enhance their functionalities and provide verifica

In [30]:
print(rag_pipeline("If I want to get votes and money in a hackathon, what themes do you recommend I tackle and with which technologies",top_k=10))

To potentially gain votes and financial backing in a hackathon, consider tackling the theme of improving donor-student interactions in education through Web3 technology. This addresses the problem of convoluted processes that hinder support for students, particularly from diverse backgrounds.

Technologies you can use for this project include:
- Solidity
- JavaScript
- React.js
- Tailwind CSS
- Gnosis Safe
- Scroll

These technologies will help in implementing a secure and effective solution that connects donors directly with students, leveraging blockchain for transparency and efficiency.
